# STQD6324 Assignment 2-P167345: MovieLens 100K Data Pipeline using Apache Spark and Cassandra

In this assignment, i builds a data pipeline using the MovieLens 100K dataset. The dataset files are loaded into HDFS, then process using Apache Spark. The results are saved into Cassandra.

Dataset files used:
- `u.data` - movie ratings
- `u.user` - user information
- `u.item` - movie information and genres

There are five task need to do:

1.Calculate the average rating for each movie

2.Identify the top 10 movies with the highest average ratings

3.Identify users who rated at least 50 movies and find their favourite genre

4.Find users who are less than 20 years old

5.Find users whose occupation is scientist and whose age is between 30 and 40

## Step 1: Create HDFS Directory

Before loading the dataset, I created a HDFS directory to store the files.

```bash
hdfs dfs -mkdir -p /user/maria_dev/assignment2
hdfs dfs -ls /user/maria_dev
```

The directory `/user/maria_dev/assignment2` was created successfully.

## Step 2: Upload Dataset into HDFS

The three dataset files were uploaded into HDFS.

```bash
hdfs dfs -ls /user/maria_dev/assignment2

I also checked the number of records in each file:
hdfs dfs -cat /user/maria_dev/assignment2/u.data | wc -l
hdfs dfs -cat /user/maria_dev/assignment2/u.user | wc -l
hdfs dfs -cat /user/maria_dev/assignment2/u.item | wc -l
```

Results:
- `u.data` = 100000 records
- `u.user` = 943 records
- `u.item` = 1682 records

All three files are uploaded and the record counts match the MovieLens 100K dataset.

## Step 3: Load Dataset and Create RDDs

After uploading the files into HDFS, I used Spark to read the raw files.

In [ ]:
#Paths used in Spark are:
ratings_path = "/user/maria_dev/assignment2/u.data"
users_path = "/user/maria_dev/assignment2/u.user"
items_path = "/user/maria_dev/assignment2/u.item"

#Created three RDDS
ratings_rdd = sc.textFile(ratings_path)
users_rdd = sc.textFile(users_path)
items_rdd = sc.textFile(items_path)

# check first 5 records
print("First 5 records from u.data:")
for row in ratings_rdd.take(5):
    print(row)

print("\nFirst 5 records from u.user:")
for row in users_rdd.take(5):
    print(row)

print("\nFirst 5 records from u.item:")
for row in items_rdd.take(5):
    print(row)

The RDDs are created from the raw files. I check first 5 records to make sure Spark can read the files correctly.

## Step 4: Transform RDDs into DataFrames

After creating the RDDs, I converted them into Spark DataFrames. so I can use Spark SQL to do the analysis.

Three DataFrames:
- `ratings_df` from `u.data` — columns: user_id, movie_id, rating, timestamp
- `users_df` from `u.user` — columns: user_id, age, gender, occupation, zip_code
- `items_df` from `u.item` — columns: movie_id, movie_title, release_date, and genre columns

In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col

# ratings_df
ratings_df = ratings_rdd.map(lambda x: x.split("\t")).map(
    lambda x: Row(user_id=int(x[0]), movie_id=int(x[1]), rating=int(x[2]), timestamp=int(x[3]))
).toDF()
ratings_df = ratings_df.select("user_id", "movie_id", "rating", "timestamp")
ratings_df.show(5)
ratings_df.printSchema()

In [ ]:
# users_df
users_df = users_rdd.map(lambda x: x.split("|")).map(
    lambda x: Row(user_id=int(x[0]), age=int(x[1]), gender=x[2], occupation=x[3], zip_code=x[4])
).toDF()
users_df.show(5)
users_df.printSchema()

In [ ]:
# movie genre columns
genre_columns = ["unknown","Action","Adventure","Animation","Children",
                 "Comedy","Crime","Documentary","Drama","Fantasy",
                 "Film_Noir","Horror","Musical","Mystery","Romance",
                 "Sci_Fi","Thriller","War","Western"]

def parse_item(line):
    parts = line.split("|")
    row = {"movie_id": int(parts[0]), "movie_title": parts[1], "release_date": parts[2]}
    for i, genre in enumerate(genre_columns):
        row[genre] = int(parts[5 + i])
    return Row(**row)

items_df = items_rdd.map(parse_item).toDF()
items_df.show(5)
items_df.printSchema()

## Step 5: Data Cleaning and Preprocessing

In this step, I checked the basic information of the DataFrames.

In [ ]:
print("Ratings count:", ratings_df.count())
print("Users count:", users_df.count())
print("Items count:", items_df.count())

In [ ]:
from pyspark.sql.functions import col, isnan, when, count

# check missing values
print("Missing values in ratings_df:")
ratings_df.select([count(when(col(c).isNull(), c)).alias(c) for c in ["user_id","movie_id","rating","timestamp"]]).show()

print("Missing values in users_df:")
users_df.select([count(when(col(c).isNull(), c)).alias(c) for c in ["user_id","age","gender","occupation","zip_code"]]).show()

print("Missing values in items_df:")
items_df.select([count(when(col(c).isNull(), c)).alias(c) for c in ["movie_id","movie_title","release_date"]]).show()

In [ ]:
# register as SQL views for Spark SQL queries
ratings_df.createOrReplaceTempView("ratings")
users_df.createOrReplaceTempView("users")
items_df.createOrReplaceTempView("items")

No missing values found. Record counts match the expected MovieLens 100K structure. The DataFrames are registered as SQL views so I can use Spark SQL for the analysis.

## Step 6:Average Rating for Each Movie

The first task is to calculate the average rating for each movie.This result helps us see how each movie was rated by users on average.

In [ ]:
from pyspark.sql.functions import avg, round as spark_round

avg_rating_df = ratings_df.groupBy("movie_id").agg(avg("rating").alias("average_rating"))

# join with items_df to get movie title
avg_rating_with_title_df = avg_rating_df.join(
    items_df.select("movie_id", "movie_title"), on="movie_id", how="left"
).select("movie_id", "movie_title", "average_rating")

avg_rating_with_title_df.show(10)

This task calculate the average rating for all 1682 movies. The result shows movie_id, movie title, and their average rating.

## Step 7: Task ii - Top 10 Movies with Highest Average Rating

In [ ]:
from pyspark.sql.functions import desc

top10_movies_df = avg_rating_with_title_df.orderBy(desc("average_rating")).limit(10)
top10_movies_df.show(10, truncate=False)

The top 10 movies all have average rating of 5.0. This is because some movies only got rated by very few people and all of them gave 5 stars. So high average rating does not always mean the movie is popular.

## Step 8: Task iii - Active Users Rated at Least 50 Movies and Their Favourite Genre

Find users who rated at least 50 movies, then find which genre they rated most.

In [ ]:
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# count how many movies each user rated
user_rating_count = ratings_df.groupBy("user_id").agg(count("movie_id").alias("rating_count"))

# filter users with at least 50 ratings
active_users = user_rating_count.filter(col("rating_count") >= 50)
active_users.show(10)
print("Total active users:", active_users.count())

In [ ]:
# join active users with ratings and items to get genre info
active_ratings = active_users.join(ratings_df, on="user_id", how="inner")
active_with_movies = active_ratings.join(items_df, on="movie_id", how="inner")
active_with_movies.show(10)

In [ ]:
from functools import reduce
from pyspark.sql import DataFrame

# convert genre columns into rows
genre_dfs = []
for genre in genre_columns:
    genre_dfs.append(
        active_with_movies.filter(col(genre) == 1)
        .select("user_id", "rating_count", lit(genre).alias("genre"))
    )

all_genres_df = reduce(DataFrame.union, genre_dfs)

# count genre frequency per user
genre_count_df = all_genres_df.groupBy("user_id", "rating_count", "genre").agg(count("genre").alias("genre_count"))
genre_count_df.show(20)

In [ ]:
# pick the genre with highest count for each user
window = Window.partitionBy("user_id").orderBy(desc("genre_count"))
favourite_genre_df = genre_count_df.withColumn("rank", row_number().over(window)).filter(col("rank") == 1).drop("rank")
favourite_genre_df.show(20)

There are 568 active users who rated at least 50 movies. Drama and Comedy are the most common favourite genres. This make sense because these two genres have the most movies in the dataset.

## Step 9: Task iv -  Users Less Than 20 Years Old

I used the `filter` function on the `users_df` DataFrame.

This result shows the younger users in the MovieLens dataset.

In [ ]:
users_under_20_df = users_df.filter(col("age") < 20)
users_under_20_df.show()
print("Total users under 20:", users_under_20_df.count())

## Step 10: Task v - Scientist Users Aged Between 30 and 40

The fifth task is to find users whose occupation is scientist and age is between 30 and 40.

I used two filter conditions:

- occupation is `scientist`
- age is between 30 and 40

This task shows how to filter users by more than one condition

In [ ]:
scientist_users_df = users_df.filter(
    (col("occupation") == "scientist") & (col("age") >= 30) & (col("age") <= 40)
).select(
    col("user_id").cast("int").alias("user_id"),
    col("age").cast("int").alias("age"),
    col("gender").cast("string").alias("gender"),
    col("occupation").cast("string").alias("occupation"),
    col("zip_code").cast("string").alias("zip_code")
)
scientist_users_df.show(20, truncate=False)
print("Total:", scientist_users_df.count())

There are 16 scientist users aged between 30 and 40. Most of them are male.

## Step 11: Create Cassandra Keyspace and Tables

After finishing all five tasks in Spark, I set up Cassandra to store the results.

I open cqlsh in PuTTY:

```bash
cd /opt/apache-cassandra-3.11.13/bin
python cqlsh.py 127.0.0.1 9042
```

Then create keyspace and tables using `cassandra_commands.sql`.

```sql
CREATE KEYSPACE IF NOT EXISTS assignment2
WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};
```

Five tables created:
- `average_movie_ratings`
- `top10_movies`
- `favourite_genres`
- `users_under_20`
- `scientist_users_30_40`

## Step 12: Export Results as CSV Files

In [ ]:
import os
import shutil
import glob

base_output_dir = "/home/maria_dev/assignment2/cassandra_csv"

if os.path.exists(base_output_dir):
    shutil.rmtree(base_output_dir)
os.makedirs(base_output_dir)

def save_single_csv(df, folder_name, final_file_name):
    temp_path = os.path.join(base_output_dir, folder_name)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv("file://" + temp_path)
    part_file = glob.glob(os.path.join(temp_path, "part-*.csv"))[0]
    final_path = os.path.join(base_output_dir, final_file_name)
    shutil.copy(part_file, final_path)
    print("CSV saved:", final_path)

save_single_csv(avg_rating_with_title_df, "average_movie_ratings_temp", "average_movie_ratings.csv")
save_single_csv(top10_movies_df, "top10_movies_temp", "top10_movies.csv")
save_single_csv(favourite_genre_df, "favourite_genres_temp", "favourite_genres.csv")
save_single_csv(users_under_20_df, "users_under_20_temp", "users_under_20.csv")
save_single_csv(scientist_users_df, "scientist_users_30_40_temp", "scientist_users_30_40.csv")

print("\nAll CSV files created.")

I tried to write Spark DataFrames directly into Cassandra but got JAR dependency error with the Spark Cassandra Connector. So I export the results as CSV files first, then use CQL COPY command to load into Cassandra.

## Step 13: Cassandra Storage
After generating the CSV files, I created one Cassandra keyspace and five Cassandra tables.

The keyspace name is:

`assignment2`

The tables are:

- `average_movie_ratings`
- `top10_movies`
- `favourite_genres`
- `users_under_20`
- `scientist_users_30_40`

Each table stores the result of one analytical task.

The CSV files were imported into Cassandra by using CQL `COPY` commands.

### Cassandra Commands Used
```sql
CREATE KEYSPACE IF NOT EXISTS assignment2

WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1};

USE assignment2;

CREATE TABLE IF NOT EXISTS average_movie_ratings (
    movie_id int PRIMARY KEY,
    movie_title text,
    average_rating double
);

CREATE TABLE IF NOT EXISTS top10_movies (
    movie_id int PRIMARY KEY,
    movie_title text,
    average_rating double
);

CREATE TABLE IF NOT EXISTS favourite_genres (
    user_id int PRIMARY KEY,
    rating_count int,
    genre text,
    genre_count int
);

CREATE TABLE IF NOT EXISTS users_under_20 (
    user_id int PRIMARY KEY,
    age int,
    gender text,
    occupation text,
    zip_code text
);

CREATE TABLE IF NOT EXISTS scientist_users_30_40 (
    user_id int PRIMARY KEY,
    age int,
    gender text,
    occupation text,
    zip_code text
);

## Step 13: Import CSV into Cassandra

Used CQL COPY command to load each CSV file into Cassandra. 

Example:

```sql
COPY top10_movies (movie_id, movie_title, average_rating)
FROM '/home/maria_dev/assignment2/cassandra_csv/top10_movies.csv'
WITH HEADER = TRUE;
```

## Step 14: Validate Cassandra Tables

After importing the CSV files, I used CQL queries to check the Cassandra tables.

First, I checked the number of records:

```sql
SELECT COUNT(*) FROM average_movie_ratings;  -- 1682
SELECT COUNT(*) FROM top10_movies;           -- 10
SELECT COUNT(*) FROM favourite_genres;       -- 568
SELECT COUNT(*) FROM users_under_20;         -- 77
SELECT COUNT(*) FROM scientist_users_30_40;  -- 16
```

All counts match the Spark results, so the data is successfully stored in Cassandra.

## Final Discussion anf Conclusion

This assignment helped me understand how a simple data pipeline works.

**Task i and ii**: I calculated average ratings for all 1682 movies. The top 10 movies all got a 5.0 rating, but only a small number of users rated them, so we need to interpret this result carefully.

**Task iii:** There were 568 users who rated no less than 50 movies. Drama and Comedy turned out to be the most liked genres for these active users.

**Task iv:**  77 users are younger than 20, and most of them are student.

**Task v:** I found 16 scientist users aged from 30 to 40, nearly all of them are male.

I ran into one annoying issue during the work: Spark Cassandra Connector came with a dependency error, so I couldn’t insert dataframe data straight into Cassandra from Spark. I ended up exporting all results as CSV files and imported them with CQL COPY command, and it worked fine with same final output.

I’ve learned the whole flow of a basic data pipeline in this coursework. Original raw data saved into HDFS, I used Spark to read and clean dataset, and Spark DataFrame lets me do filter, join, grouping and sorting to finish all analysis tasks.